<div style="text-align:center; border-radius:15px; padding:15px; color:white; margin:0; font-family: 'Orbitron', sans-serif; background: #2E0249; background: #11001C; box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.3); overflow:hidden; margin-bottom: 1em;">
  <div style="font-size:150%; color:#FEE100"><b>Paris Housing Prices Data Analysis</b></div>
  <div>This notebook was created with the help of <a href="https://devra.ai/ref/kaggle" style="color:#6666FF">Devra AI</a></div>
</div>

Housing in Paris always comes with a touch of romance and mystery. This dataset not only represents the diverse architectural fingerprint of Paris but also presents a unique opportunity to analyze how different features, such as property type, location, and building characteristics, influence housing prices in the City of Lights.

If you enjoy this analysis, please consider upvoting it.

# Table of Contents

- [Data Loading](#Data-Loading)
- [Data Cleaning and Preprocessing](#Data-Cleaning-and-Preprocessing)
- [Exploratory Data Analysis (EDA)](#Exploratory-Data-Analysis-EDA)
- [Correlation Analysis](#Correlation-Analysis)
- [Feature Engineering and Prediction](#Feature-Engineering-and-Prediction)
- [Conclusions and Future Work](#Conclusions-and-Future-Work)

In [ ]:
# Import necessary libraries and suppress warnings
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')  # Ensure matplotlib is set to use non-interactive backend
import matplotlib.pyplot as plt
plt.switch_backend('Agg')  # switch backend if necessary

import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Enable inline plotting
%matplotlib inline


# Data Loading

We start by loading the Paris housing prices dataset. The dataset contains various features like property type, location details, and building characteristics. The file is encoded in ASCII and uses commas as delimiters.

In [ ]:
# Load the dataset
data_file = 'paris_housing_prices_dataset.csv'
df = pd.read_csv(data_file, delimiter=',', encoding='ascii')

# Display the first few rows of the dataframe
df.head()

# Data Cleaning and Preprocessing

Before we dive into exploratory analysis, we need to ensure that the data is clean and correctly formatted. Notably, although the dataset description does not specify any dates, there is an implicit assumption that the 'Year_Built' is a year and should be treated as numeric. We also check for missing values and any obvious inconsistencies.

In [ ]:
# Check data info
df.info()

# Check for missing values and basic statistics
print(df.isnull().sum())
print(df.describe(include='all'))

# Convert any columns if needed (in this dataset, 'Year_Built' is already numeric)
# If specific date columns were present, we'd infer their datetime type accordingly


# Exploratory Data Analysis (EDA)

Here we explore the dataset to understand the distribution of different features. We will use various visualization techniques including histograms, pie charts, barplots, violin plots, and box plots to uncover insights into the Paris housing market.

In [ ]:
# Set up default figure size
plt.figure(figsize=(10, 6))

# Histogram for Size (sqm)
sns.histplot(df['Size_sqm'], kde=True)
plt.title('Distribution of Property Size (sqm)')
plt.xlabel('Size (sqm)')
plt.ylabel('Frequency')
plt.show()

# Pie chart equivalent using countplot for Property Type
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='Property_Type')
plt.title('Count of Property Types')
plt.xlabel('Property Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

# Box Plot for Price
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, y='Price_EUR')
plt.title('Box Plot for Price (EUR)')
plt.ylabel('Price (EUR)')
plt.show()

# Violin and Boxen Plot for Rooms
plt.figure(figsize=(10, 6))
sns.violinplot(data=df, x='Rooms')
plt.title('Violin Plot for Number of Rooms')
plt.xlabel('Rooms')
plt.show()

plt.figure(figsize=(10, 6))
sns.boxenplot(data=df, x='Rooms')
plt.title('Boxen Plot for Number of Rooms')
plt.xlabel('Rooms')
plt.show()

# Pair plot to see relationships between some numeric features
numeric_features = ['Size_sqm', 'Rooms', 'Floor', 'Year_Built', 'Distance_to_Center_km', 'Price_EUR']
sns.pairplot(df[numeric_features])
plt.show()


# Correlation Analysis

Next, we look at the correlation among the numeric features. This can help us uncover relationships and possible collinearity among predictors. Note that we consider only numeric columns.

In [ ]:
# Select only numeric features
numeric_df = df.select_dtypes(include=[np.number])

# If there are four or more numeric columns, plot a heatmap
if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(12, 8))
    corr_matrix = numeric_df.corr()
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
    plt.title('Correlation Heatmap of Numeric Features')
    plt.show()
else:
    print('Not enough numeric features to perform correlation heatmap.')


# Feature Engineering and Prediction

In this section, we build a simple predictive model to estimate the housing price in EUR (Price_EUR) from the other available features. We use a Linear Regression model since housing price is a continuous variable. The features include both numeric and categorical values. For simplicity, we focus on a subset of predictors after applying minimal feature engineering.

Note: If you encounter errors related to nondeterministic numeric conversions or unexpected categories during encoding, be sure to check the encoding of your dataset and ensure that categorical variables are properly transformed.

In [ ]:
# Prepare data for prediction

# Select features and target variable
# We'll use numeric features and one-hot encode the categorical ones
features = ['Arrondissement', 'Property_Type', 'Size_sqm', 'Rooms', 'Floor', 'Year_Built', 'Distance_to_Center_km']
target = 'Price_EUR'

df_model = df[features + [target]].copy()

# One-hot encode the 'Property_Type' categorical variable
df_model = pd.get_dummies(df_model, columns=['Property_Type'], drop_first=True)

# Separate features and target
X = df_model.drop(columns=[target])
y = df_model[target]

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build and train a Linear Regression model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict on the test set
y_pred = lr_model.predict(X_test)

# Calculate prediction accuracy (R^2 score)
r2 = r2_score(y_test, y_pred)
print(f'Linear Regression R^2 Score: {r2:.4f}')

# Plot the predicted vs actual values
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.xlabel('Actual Price (EUR)')
plt.ylabel('Predicted Price (EUR)')
plt.title('Actual vs Predicted Housing Prices')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', lw=2)
plt.show()


# Conclusions and Future Work

The analysis presented in this notebook provides a foundational exploration of the Paris housing dataset. We performed data cleaning, explored feature distributions using a wide array of plots, and built a linear regression model to predict housing prices.

Merits of this approach include:
- Comprehensive visualization methods that offer multiple perspectives on the data
- A simple predictive modeling pipeline that can be refined or extended
- Consideration of common errors related to date and categorical data processing

Future analysis could involve:
- More sophisticated feature engineering, including interactions between variables
- Testing alternative machine learning algorithms for improved prediction accuracy
- Incorporating spatial analysis if geolocation data becomes available
- Applying cross-validation techniques to further validate model performance

Thank you for reviewing this analysis. If you found it useful, please consider upvoting.